# ExploreKit — end-to-end demo

Notebook только демонстрирует готовую библиотеку. Основная логика остаётся в `.py` модулях.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
ROOT


## 1. Smoke: policy → DecisionLog → OPE

Используется реальная epsilon-greedy политика участника 1, логирование участника 4 и IPS участника 2.

In [ ]:
from explorekit.integration import run_smoke

smoke = run_smoke(seed=42, n_rounds=1000)
{
    "n_logs": smoke.n_logs,
    "mean_reward": smoke.mean_reward,
    "ips": smoke.ips_estimate,
    "ess": smoke.ess,
    "reliability": smoke.reliability,
}


## 2. End-to-end experiment

Для notebook уменьшаем размер только ради быстрого демонстрационного запуска; параметры финального CLI остаются в YAML.

In [ ]:
from experiments.run_experiment import run_experiment
from explorekit.integration import load_config

config = load_config(ROOT / "configs" / "moderate.yaml")
config["simulation"]["n_rounds"] = 1000
config["bootstrap"]["n_bootstrap"] = 100
result = run_experiment(config, save=False)
result


In [ ]:
import pandas as pd

pd.DataFrame([{
    "preset": result.preset_name,
    "policy": result.evaluation_policy,
    "true_ctr": result.true_ctr,
    "ips": result.ips_estimate,
    "snips": result.snips_estimate,
    "dr": result.dr_estimate,
    "ess": result.ess,
    "reliability": result.reliability,
    "exploration_share": result.exploration_share,
    "cold_item_speedup": result.cold_item_speedup,
    "ctr_cost": result.ctr_cost,
}])


## 3. Симулятор и Cold-Item Booster

Каталог, скрытая вероятность клика и сравнение пресетов живут в `explorekit.simulator`. Notebook только вызывает готовые функции.

In [ ]:
from explorekit.simulator import ItemCatalog, make_replication
from explorekit.policies import EpsilonGreedyPolicy
from experiments.cold_item_experiment import run_cold_item_experiment

rng = __import__("numpy").random.default_rng(42)
catalog = ItemCatalog(n_actions=12, feature_dim=4, rng=rng)
print("warm items:", catalog.n_warm)
print("available day 0:", catalog.available_at(0).tolist())
print("available day 10:", catalog.available_at(10).tolist())

policy = EpsilonGreedyPolicy(
    n_actions=12, epsilon=0.1, min_epsilon=0.0, max_epsilon=1.0, seed=42
)
rep = make_replication(seed=42, n_rounds=200, policy=policy, time_to_n_threshold=8)
print({
    "true_ctr": round(rep.true_value, 4),
    "speedup": None if rep.cold_item_speedup is None else round(rep.cold_item_speedup, 3),
    "ctr_cost": None if rep.ctr_cost is None else round(rep.ctr_cost, 4),
    "n_logs": len(rep.logs),
    "chosen_item_type": type(rep.logs[0].chosen_item).__name__,
})

run_cold_item_experiment(n_rounds=160)